# 🌍 EDA: Environment & Sustainability in AI

**Project:** Gemma 4 - Good AI for Humanity, Not Just Performance  
**Notebook:** Exploratory Data Analysis on Environment-Themed AI Responses  
**Author:** Data Science Team  
**Last Updated:** 2024-02-15

## 📋 Table of Contents
1. [Introduction](#introduction)
2. [Data Loading & Preprocessing](#data-loading)
3. [Distribution Analysis](#distribution-analysis)
4. [Score Correlation Analysis](#correlation-analysis)
5. [Text Length vs Quality](#text-length-analysis)
6. [Keyword Analysis](#keyword-analysis)
7. [Environment vs Other Categories](#category-comparison)
8. [Key Findings & Insights](#findings)

---
## 1. Introduction <a name="introduction"></a>

This notebook explores the **environment and sustainability** category within our AI response dataset. We aim to understand:
- How AI responses about environmental topics perform across quality dimensions
- Whether environment-themed responses score differently on humanity metrics
- Common themes and keywords in high-scoring environmental AI responses
- Patterns that distinguish good environmental AI responses from poor ones

In [ ]:
# ============================================
# Cell 1: Import Libraries
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

---
## 2. Data Loading & Preprocessing <a name="data-loading"></a>

In [ ]:
# ============================================
# Cell 2: Load Datasets
# ============================================
DATA_DIR = '../data/raw/'

train_df = pd.read_csv(f'{DATA_DIR}train.csv')
test_df = pd.read_csv(f'{DATA_DIR}test.csv')
labels_df = pd.read_csv(f'{DATA_DIR}labels.csv')
corpus_text = open(f'{DATA_DIR}corpus.txt', 'r', encoding='utf-8').read()

print(f"Train set: {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test set:  {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"Labels:    {labels_df.shape[0]} rows, {labels_df.shape[1]} columns")
print(f"Corpus:    {len(corpus_text)} characters")

In [ ]:
# ============================================
# Cell 3: Filter Environment Category
# ============================================
env_df = train_df[train_df['category'] == 'environment'].copy()
non_env_df = train_df[train_df['category'] != 'environment'].copy()

print(f"Environment records: {len(env_df)}")
print(f"Non-environment records: {len(non_env_df)}")
print(f"\nEnvironment data sample:")
env_df.head()

In [ ]:
# ============================================
# Cell 4: Add Derived Features
# ============================================
def add_text_features(df):
    df['prompt_length'] = df['prompt'].str.len()
    df['response_length'] = df['response'].str.len()
    df['response_word_count'] = df['response'].str.split().str.len()
    df['prompt_word_count'] = df['prompt'].str.split().str.len()
    df['avg_word_length'] = df['response'].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if pd.notna(x) else 0
    )
    df['sentence_count'] = df['response'].apply(
        lambda x: len(re.split(r'[.!?]+', str(x))) if pd.notna(x) else 0
    )
    df['composite_score'] = (
        df['humanity_score'] * 0.35 +
        df['performance_score'] * 0.25 +
        df['helpfulness_score'] * 0.40
    )
    return df

env_df = add_text_features(env_df)
non_env_df = add_text_features(non_env_df)

print("Derived features added:")
print(env_df[['response_word_count', 'sentence_count', 'composite_score']].describe())

---
## 3. Distribution Analysis <a name="distribution-analysis"></a>

In [ ]:
# ============================================
# Cell 5: Score Distributions for Environment Category
# ============================================
score_cols = ['humanity_score', 'performance_score', 'helpfulness_score']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c']

for idx, (col, color) in enumerate(zip(score_cols, colors)):
    axes[idx].hist(env_df[col], bins=10, color=color, alpha=0.7, edgecolor='white')
    axes[idx].axvline(env_df[col].mean(), color='black', linestyle='--', linewidth=2,
                      label=f'Mean: {env_df[col].mean():.2f}')
    axes[idx].set_title(f'Distribution of {col.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Score')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.suptitle('Score Distributions for Environment Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_env_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# Cell 6: Difficulty Distribution
# ============================================
fig, ax = plt.subplots(figsize=(8, 5))
difficulty_counts = env_df['difficulty'].value_counts()
bars = ax.bar(difficulty_counts.index, difficulty_counts.values,
              color=['#27ae60', '#f39c12', '#e74c3c'], alpha=0.85, edgecolor='white')

for bar, val in zip(bars, difficulty_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Environment Prompts by Difficulty Level', fontsize=14, fontweight='bold')
ax.set_xlabel('Difficulty')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../data/processed/eda_env_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Score Correlation Analysis <a name="correlation-analysis"></a>

In [ ]:
# ============================================
# Cell 7: Correlation Heatmap
# ============================================
numeric_cols = ['humanity_score', 'performance_score', 'helpfulness_score',
                'response_word_count', 'sentence_count', 'composite_score']

corr_matrix = env_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'})
ax.set_title('Correlation Matrix: Environment Category', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/processed/eda_env_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Correlations with composite_score:")
print(corr_matrix['composite_score'].sort_values(ascending=False))

---
## 5. Text Length vs Quality <a name="text-length-analysis"></a>

In [ ]:
# ============================================
# Cell 8: Response Word Count vs Composite Score
# ============================================
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(env_df['response_word_count'], env_df['composite_score'],
           c=env_df['humanity_score'], cmap='RdYlGn', s=100, alpha=0.8, edgecolors='gray')

# Trend line
z = np.polyfit(env_df['response_word_count'], env_df['composite_score'], 1)
p = np.poly1d(z)
x_line = np.linspace(env_df['response_word_count'].min(), env_df['response_word_count'].max(), 100)
ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2, label=f'Trend (slope={z[0]:.4f})')

ax.set_title('Response Length vs Composite Quality Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Response Word Count')
ax.set_ylabel('Composite Score')
ax.legend(loc='best')
cbar = plt.colorbar(ax.collections[0], ax=ax, label='Humanity Score')
plt.tight_layout()
plt.savefig('../data/processed/eda_env_length_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Keyword Analysis <a name="keyword-analysis"></a>

In [ ]:
# ============================================
# Cell 9: Extract Top Keywords from Environment Responses
# ============================================
stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
              'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
              'should', 'may', 'might', 'can', 'shall', 'to', 'of', 'in', 'for',
              'on', 'with', 'at', 'by', 'from', 'as', 'into', 'through', 'during',
              'before', 'after', 'above', 'below', 'between', 'and', 'but', 'or',
              'nor', 'not', 'so', 'yet', 'both', 'either', 'neither', 'each',
              'every', 'all', 'any', 'few', 'more', 'most', 'other', 'some',
              'such', 'no', 'only', 'own', 'same', 'than', 'too', 'very',
              'just', 'because', 'if', 'when', 'where', 'how', 'what', 'which',
              'who', 'whom', 'this', 'that', 'these', 'those', 'it', 'its', 'we',
              'they', 'their', 'them', 'our', 'us', 'you', 'your', 'he', 'she',
              'his', 'her', 'also', 'about', 'up', 'out', 'while', 'like'}

def extract_keywords(texts, top_n=30):
    words = []
    for text in texts:
        words.extend(re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower()))
    filtered = [w for w in words if w not in stop_words]
    return Counter(filtered).most_common(top_n)

env_keywords = extract_keywords(env_df['response'].tolist())
print("Top 30 Keywords in Environment Responses:")
for word, count in env_keywords:
    print(f"  {word:25s} {count:4d}")

In [ ]:
# ============================================
# Cell 10: Keyword Frequency Bar Chart
# ============================================
top_20 = env_keywords[:20]
words = [w for w, c in top_20]
counts = [c for w, c in top_20]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(words)), counts, color='#27ae60', alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Keywords in Environment AI Responses', fontsize=14, fontweight='bold')

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/eda_env_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Environment vs Other Categories <a name="category-comparison"></a>

In [ ]:
# ============================================
# Cell 11: Category Comparison Box Plot
# ============================================
all_df = add_text_features(train_df.copy())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['humanity_score', 'performance_score', 'helpfulness_score']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    categories = sorted(all_df['category'].unique())
    data_groups = [all_df[all_df['category'] == cat][metric].values for cat in categories]
    bp = axes[idx].boxplot(data_groups, labels=categories, patch_artist=True)
    
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    axes[idx].axhline(all_df[metric].mean(), color='red', linestyle=':', alpha=0.5,
                       label=f'Overall Mean: {all_df[metric].mean():.2f}')
    axes[idx].set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Score')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].legend(loc='best')

plt.suptitle('Score Comparison Across Categories', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/eda_env_category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# Cell 12: Category Mean Scores Summary
# ============================================
category_summary = all_df.groupby('category').agg(
    count=('id', 'count'),
    avg_humanity=('humanity_score', 'mean'),
    avg_performance=('performance_score', 'mean'),
    avg_helpfulness=('helpfulness_score', 'mean'),
    avg_composite=('composite_score', 'mean'),
    avg_word_count=('response_word_count', 'mean')
).round(3)

category_summary = category_summary.sort_values('avg_composite', ascending=False)
print("Category Performance Summary:")
print(category_summary.to_string())

---
## 8. Key Findings & Insights <a name="findings"></a>

In [ ]:
# ============================================
# Cell 13: Summary Statistics
# ============================================
print("=" * 60)
print("ENVIRONMENT EDA - KEY FINDINGS")
print("=" * 60)

print(f"\n1. Dataset Overview:")
print(f"   - Total environment records: {len(env_df)}")
print(f"   - Average humanity score:    {env_df['humanity_score'].mean():.2f}")
print(f"   - Average performance score: {env_df['performance_score'].mean():.2f}")
print(f"   - Average helpfulness score: {env_df['helpfulness_score'].mean():.2f}")
print(f"   - Average composite score:   {env_df['composite_score'].mean():.2f}")

print(f"\n2. Text Characteristics:")
print(f"   - Avg response length: {env_df['response_word_count'].mean():.0f} words")
print(f"   - Avg sentences:       {env_df['sentence_count'].mean():.0f}")
print(f"   - Avg word length:     {env_df['avg_word_length'].mean():.1f} characters")

print(f"\n3. Top 5 Keywords:")
for word, count in env_keywords[:5]:
    print(f"   - '{word}' (appears {count} times)")

print(f"\n4. Environment vs Overall:")
print(f"   - Humanity:    {env_df['humanity_score'].mean():.2f} vs {all_df['humanity_score'].mean():.2f} overall")
print(f"   - Performance: {env_df['performance_score'].mean():.2f} vs {all_df['performance_score'].mean():.2f} overall")
print(f"   - Helpfulness: {env_df['helpfulness_score'].mean():.2f} vs {all_df['helpfulness_score'].mean():.2f} overall")

In [ ]:
# ============================================
# Cell 14: Save Processed Data
# ============================================
env_df.to_csv('../data/processed/eda_environment_results.csv', index=False)
category_summary.to_csv('../data/processed/eda_category_summary.csv')

print("Processed data saved successfully!")
print("  - ../data/processed/eda_environment_results.csv")
print("  - ../data/processed/eda_category_summary.csv")

---
## ✅ Notebook Complete

**Next Steps:**
- Run `eda_health.ipynb` for healthcare-focused analysis
- Compare environment findings with health category patterns
- Use insights to inform feature engineering in `baseline_model.ipynb`